# **Lab 5: Build and Train LSTM for Text Classification & Sequence-to-Sequence**
This notebook demonstrates:
1. LSTM for Text Classification (IMDB Dataset)
2. Sequence-to-Sequence (Encoder-Decoder LSTM)


# **Text Classification Architectures**



| Layer / Step      | **RNN**                       | **LSTM**                      | **GRU**                       |
|------------------|-------------------------------|-------------------------------|-------------------------------|
| **Input**         | Text sequence (`max_len`)     | Text sequence (`max_len`)     | Text sequence (`max_len`)     |
| **Tokenization**  | Words → integers              | Words → integers              | Words → integers              |
| **Padding**       | Pad to `max_len`              | Pad to `max_len`              | Pad to `max_len`              |
| **Embedding**     | `Embedding(input_dim, output_dim, input_length)` | `Embedding(input_dim, output_dim, input_length)` | `Embedding(input_dim, output_dim, input_length)` |
| **Recurrent**     | `SimpleRNN(32)`               | `LSTM(64)`                    | `GRU(64)`                     |
| **Dropout**       | `Dropout(0.5)`                | `Dropout(0.5)`                | `Dropout(0.5)`                |
| **Output**        | `Dense(1, activation='sigmoid')` | `Dense(1, activation='sigmoid')` | `Dense(1, activation='sigmoid')` |
| **Loss Function** | Binary cross-entropy           | Binary cross-entropy           | Binary cross-entropy           |
| **Optimizer**     | Adam                          | Adam                          | Adam                          |
| **Metrics**       | Accuracy                       | Accuracy                       | Accuracy                       |

# **Built-in Keras Recurrent Layers**

| Layer        | Built-in Keras Layer                                      | Key Parameters |
|-------------|-----------------------------------------------------------|----------------|
| **SimpleRNN** | `tf.keras.layers.SimpleRNN(units=32, activation='tanh', return_sequences=False, dropout=0.0, recurrent_dropout=0.0)` | units, activation, return_sequences, dropout, recurrent_dropout |
| **LSTM**     | `tf.keras.layers.LSTM(units=64, activation='tanh', recurrent_activation='sigmoid', return_sequences=False, dropout=0.0, recurrent_dropout=0.0)` | units, activation, recurrent_activation, return_sequences, dropout, recurrent_dropout |
| **GRU**      | `tf.keras.layers.GRU(units=64, activation='tanh', recurrent_activation='sigmoid', return_sequences=False, dropout=0.0, recurrent_dropout=0.0)` | units, activation, recurrent_activation, return_sequences, dropout, recurrent_dropout |

.

| Parameter / Layer        | **SimpleRNN**                                                                                  | **LSTM**                                                                                                      | **GRU**                                                                                                      |
|--------------------------|------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------|-------------------------------------------------------------------------------------------------------------|
| **Layer**                | `tf.keras.layers.SimpleRNN(...)`                                                               | `tf.keras.layers.LSTM(...)`                                                                                  | `tf.keras.layers.GRU(...)`                                                                                  |
| **Units**                | 32                                                                                             | 64                                                                                                          | 64                                                                                                          |
| **Activation**           | `tanh`                                                                                         | `tanh`                                                                                                      | `tanh`                                                                                                      |
| **Recurrent Activation** | None (no gates)                                                                                 | `sigmoid`                                                                                                   | `sigmoid`                                                                                                   |
| **Return Sequences**     | `False`                                                                                        | `False`                                                                                                     | `False`                                                                                                     |
| **Dropout**              | 0.0                                                                                            | 0.0                                                                                                         | 0.0                                                                                                         |
| **Recurrent Dropout**    | 0.0                                                                                            | 0.0                                                                                                         | 0.0                                                                                                         |

# **BASIC RNN(RECURRENT NEURAL NETWORK)**

 **RNN PIPELINE -**

 **`Raw text`** → **`Tokenization & Padding`** → **`Train/Test Split`** → **`Embedding`** → **`SimpleRNN`** → **`Dropout`** → **`Dense`** → **`Train`** → **`Evaluate`** → **`Predict Sentiment`**

**Step 1: Import Libraries**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from sklearn.model_selection import train_test_split

**Step 2: Sample Data**

In [ ]:
texts = [
    "I love this product",
    "This is a bad movie",
    "Excellent book",
    "Worst experience ever",
    "I feel great",
    "I hate it",
    "I hate this movie but ending is good",  # mixed sentiment
    "The movie was boring but visuals were amazing"
]

labels = [1, 0, 1, 0, 1, 0, 1, 1]  # 1 = positive, 0 = negative (label mixed sentences as positive for simplicity)

**Step 3: Tokenize and Pad Sequences**

In [ ]:
max_words = 1000                   # Maximum number of words to keep in vocabulary  --- Only top 1000 most frequent words will be considered
max_len = 15                       # Maximum length of each sequence (sentence) ----  All sequences will be padded/truncated to length 15

# Create tokenizer
# oov_token = "<OOV>" will replace unknown words
tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")       # num_words → limit vocab size to 1000        # oov_token → replace unknown words with "<OOV>"

# Fit tokenizer on texts
# Example:
# "students of aimlc are brilliant"
# → {'students':2, 'of':3, 'aimlc':4, 'are':5, 'brilliant':6}
tokenizer.fit_on_texts(texts)                                       # Assigns a unique integer to each word

# Convert texts to sequences
# Example:
# "students of aimlc are brilliant"
# → [2, 3, 4, 5, 6]
sequences = tokenizer.texts_to_sequences(texts)                     # Convert each sentence into sequence of integers

# Pad sequences to ensure same length (15)
# padding='post' → add zeros at the end if sentence is short
# Longer sentences will be truncated
# Apply padding
# max_len = 10
# padding='post' → add zeros at end
# Example:
# [2, 3, 4, 5, 6]
# → [2, 3, 4, 5, 6, 0, 0, 0, 0, 0]
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

# Print example output
# Example output:
# "students of aimlc are brilliant"
# → [2 3 4 5 6 0 0 0 0 0]
print("Example padded sequence:\n", padded_sequences[2])

Example padded sequence:
 [12 13  0  0  0  0  0  0  0  0  0  0  0  0  0]


**Step 4: Train-Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.25, random_state=42               # test_size=0.25 → 25% data for testing, 75% for training.
                                                                            # random_state=42 → ensures same split every time (reproducibility)
)
# padded_sequences =
# [
#   [2, 3, 4, 0, 0],
#   [5, 6, 0, 0, 0],
#   [7, 8, 9, 0, 0],
#   [1, 2, 3, 4, 5]
# ]

# labels = [1, 0, 1, 0]
# After split (75% train, 25% test):

# X_train =
# [
#   [2, 3, 4, 0, 0],
#   [5, 6, 0, 0, 0],
#   [7, 8, 9, 0, 0]
# ]

# y_train = [1, 0, 1]

# X_test =
# [
#   [1, 2, 3, 4, 5]
# ]
# y_test = [0]


**Step 5: Build RNN Model**

In [ ]:
embedding_dim = 50                               # Define embedding dimension (size of word vector representation)

rnn_model = Sequential([

    # Embedding Layer:
    # Converts word indices into dense vectors of fixed size (50 here)
    # input_dim = vocabulary size (max_words)
    # output_dim = embedding size (50)
    # input_length = length of each input sequence
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),

    # Simple RNN Layer:
    # 32 = number of hidden units (neurons)
    # Processes sequence step-by-step and captures temporal dependencies
    SimpleRNN(32),

    Dropout(0.5),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
rnn_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**Step 6: Train the Model**

In [ ]:
rnn_history = rnn_model.fit(
    X_train, np.array(y_train),
    epochs=15,
    batch_size=2,
    validation_data=(X_test, np.array(y_test))
)

Epoch 1/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 165ms/step - accuracy: 0.8333 - loss: 0.6584 - val_accuracy: 0.0000e+00 - val_loss: 0.8736
Epoch 2/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 33ms/step - accuracy: 0.8333 - loss: 0.5152 - val_accuracy: 0.0000e+00 - val_loss: 0.9621
Epoch 3/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.8333 - loss: 0.4435 - val_accuracy: 0.0000e+00 - val_loss: 1.0616
Epoch 4/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8333 - loss: 0.4336 - val_accuracy: 0.0000e+00 - val_loss: 1.1658
Epoch 5/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8333 - loss: 0.4690 - val_accuracy: 0.0000e+00 - val_loss: 1.2168
Epoch 6/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 1.0000 - loss: 0.3460 - val_accuracy: 0.0000e+00 - val_loss: 1.2873
Epoch 7/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 41ms/step - accuracy: 0.8333 - loss: 0.3580 - val_accuracy: 0.0000e+00 - val_loss: 1.3339
Epoch 8/15
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8333 - loss: 0.2879 - val_accurac

**Step 7: Test a New Sentence**

In [ ]:
new_text = ["I hate this movie but the ending is very bad"]

seq = tokenizer.texts_to_sequences(new_text)
padded_seq = pad_sequences(seq, maxlen=max_len, padding='post')
pred = rnn_model.predict(padded_seq)

print(f"Sentiment score: {pred[0][0]:.2f}")  # >0.5 → positive, <0.5 → negative

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
Sentiment score: 0.96


# **LSTM LONG SHORT TERM MEMORY**

**LSTM PIPELINE -**

**`Raw text`** → **`Tokenization & Padding`** → **`Train/Test Split`** → **`Embedding`** → **`LSTM`** → **`Dropout`** → **`Dense`** → **`Train`** → **`Evaluate`** → **`Predict Sentiment`**  


**Step 1: Import Libraries**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split

**Step 2: Prepare Dataset**

In [ ]:
texts = [
    "I love this product",
    "This is a bad movie",
    "Excellent book",
    "Worst experience ever",
    "I feel great",
    "I hate it",
    "I hate this movie but ending is good",  # mixed sentiment
    "The movie was boring but visuals were amazing"
]

labels = [1, 0, 1, 0, 1, 0, 1, 1]  # 1 = positive, 0 = negative

**Step 3: Tokenize and Pad Sequences**

In [ ]:
max_words = 1000
max_len = 15

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)
sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

print("Example padded sequence:\n", padded_sequences[6])

Example padded sequence:
 [ 2  6  3  4  7 20  5 21  0  0  0  0  0  0  0]


**Step 4: Train-Test Split**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.25, random_state=42
)

**Step 5: Build LSTM Model**

In [ ]:
embedding_dim = 50

lstm_model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    LSTM(64),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # binary classification
])

lstm_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
lstm_model.summary()

Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

 **Step 6: Train the Model**

In [ ]:
lstm_history = lstm_model.fit(
    X_train, np.array(y_train),
    epochs=20,
    batch_size=2,
    validation_data=(X_test, np.array(y_test))
)

Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 3s 174ms/step - accuracy: 0.5000 - loss: 0.6903 - val_accuracy: 0.0000e+00 - val_loss: 0.7240
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8333 - loss: 0.6770 - val_accuracy: 0.0000e+00 - val_loss: 0.7712
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8333 - loss: 0.6495 - val_accuracy: 0.0000e+00 - val_loss: 0.8229
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.8333 - loss: 0.6022 - val_accuracy: 0.0000e+00 - val_loss: 0.8771
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8333 - loss: 0.5945 - val_accuracy: 0.0000e+00 - val_loss: 0.9504
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.8333 - loss: 0.5514 - val_accuracy: 0.0000e+00 - val_loss: 1.0812
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8333 - loss: 0.5486 - val_accuracy: 0.0000e+00 - val_loss: 1.2474
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8333 - loss: 0.4716 - val_accurac

**Step 7: Evaluate**

In [ ]:
loss, accuracy = lstm_model.evaluate(X_test, np.array(y_test))
#print(f"Test Accuracy: {accuracy*100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 623ms/step - accuracy: 0.0000e+00 - loss: 1.9748


**Step 8: Predict New Sentiment**

In [ ]:
new_texts = [
    "I hate this movie but the ending is good",
    "Absolutely loved the story and characters"
]

seq = tokenizer.texts_to_sequences(new_texts)
padded_seq = pad_sequences(seq, maxlen=max_len, padding='post')
predictions = lstm_model.predict(padded_seq)

for text, pred in zip(new_texts, predictions):
    sentiment = "Positive" if pred[0] > 0.5 else "Negative"
    print(f"Text: {text}\nPredicted sentiment: {sentiment}\n")

NameError: name 'lstm_model' is not defined

# **GRU GATED RECURRENT UNIT**


**GRU PIPELINE -**

**`Raw text`** → **`Tokenization & Padding`** → **`Train/Test Split`** → **`Embedding`** → **`GRU`** → **`Dropout`** → **`Dense`** → **`Train`** → **`Evaluate`** → **`Predict Sentiment`**  

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GRU, Dense, Dropout
from sklearn.model_selection import train_test_split

In [ ]:
texts = [
    "I love this product",
    "This is a bad movie",
    "Excellent book",
    "Worst experience ever",
    "I feel great",
    "I hate it",
    "I hate this movie but ending is good",  # mixed sentiment
    "The movie was boring but visuals were amazing"
]

labels = [1, 0, 1, 0, 1, 0, 1, 1]  # 1 = positive, 0 = negative

In [ ]:
max_words = 1000
max_len = 15

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(texts)

sequences = tokenizer.texts_to_sequences(texts)
padded_sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    padded_sequences, labels, test_size=0.25, random_state=42
)

In [ ]:
embedding_dim = 50

gru_model = Sequential([
    Embedding(input_dim=max_words, output_dim=embedding_dim, input_length=max_len),
    GRU(64),
    Dropout(0.5),
    Dense(1, activation='sigmoid')  # binary classification
])

gru_model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
gru_model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_9 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ gru_1 (GRU)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_9 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_9 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
gru_history = gru_model.fit(
    X_train, np.array(y_train),
    epochs=20,
    batch_size=2,
    validation_data=(X_test, np.array(y_test))
)

Epoch 1/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 190ms/step - accuracy: 0.5000 - loss: 0.6995 - val_accuracy: 0.0000e+00 - val_loss: 0.6938
Epoch 2/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.6667 - loss: 0.6905 - val_accuracy: 0.0000e+00 - val_loss: 0.7380
Epoch 3/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.8333 - loss: 0.6450 - val_accuracy: 0.0000e+00 - val_loss: 0.7752
Epoch 4/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8333 - loss: 0.6346 - val_accuracy: 0.0000e+00 - val_loss: 0.8257
Epoch 5/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.8333 - loss: 0.6060 - val_accuracy: 0.0000e+00 - val_loss: 0.8758
Epoch 6/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - accuracy: 0.8333 - loss: 0.5690 - val_accuracy: 0.0000e+00 - val_loss: 0.9395
Epoch 7/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8333 - loss: 0.5794 - val_accuracy: 0.0000e+00 - val_loss: 1.0292
Epoch 8/20
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 35ms/step - accuracy: 0.8333 - loss: 0.5089 - val_accurac

In [ ]:
loss, accuracy = gru_model.evaluate(X_test, np.array(y_test))
#print(f"Test Accuracy: {accuracy*100:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 523ms/step - accuracy: 0.0000e+00 - loss: 1.8138


In [ ]:
new_texts = [
    "I hate this movie but the ending is good",
    "Absolutely loved the story and characters"
]

seq = tokenizer.texts_to_sequences(new_texts)
padded_seq = pad_sequences(seq, maxlen=max_len, padding='post')
predictions = gru_model.predict(padded_seq)

for text, pred in zip(new_texts, predictions):
    sentiment = "Positive" if pred[0] > 0.5 else "Negative"
    print(f"Text: {text}\nPredicted sentiment: {sentiment}\n")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 52ms/step
Text: I hate this movie but the ending is good
Predicted sentiment: Positive

Text: Absolutely loved the story and characters
Predicted sentiment: Positive



# **RNN - LSTM - GRU**

In [ ]:
print("\n=== FINAL COMPARISON ===")

print(f"RNN  -> Train Acc: {rnn_history.history['accuracy'][-1]:.4f}, "
      f"Loss: {rnn_history.history['loss'][-1]:.4f}, "
      f"Val Acc: {rnn_history.history['val_accuracy'][-1]:.4f}, "
      f"Val Loss: {rnn_history.history['val_loss'][-1]:.4f}")

print(f"LSTM -> Train Acc: {lstm_history.history['accuracy'][-1]:.4f}, "
      f"Loss: {lstm_history.history['loss'][-1]:.4f}, "
      f"Val Acc: {lstm_history.history['val_accuracy'][-1]:.4f}, "
      f"Val Loss: {lstm_history.history['val_loss'][-1]:.4f}")

print(f"GRU  -> Train Acc: {gru_history.history['accuracy'][-1]:.4f}, "
      f"Loss: {gru_history.history['loss'][-1]:.4f}, "
      f"Val Acc: {gru_history.history['val_accuracy'][-1]:.4f}, "
      f"Val Loss: {gru_history.history['val_loss'][-1]:.4f}")


=== FINAL COMPARISON ===
RNN  -> Train Acc: 1.0000, Loss: 0.0706, Val Acc: 0.0000, Val Loss: 1.8885
LSTM -> Train Acc: 0.8333, Loss: 0.3178, Val Acc: 0.0000, Val Loss: 1.9748
GRU  -> Train Acc: 0.8333, Loss: 0.4349, Val Acc: 0.0000, Val Loss: 1.8138


# **LSTM - IMDB DATASET**

**Step 1: Import Libraries**

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout ,Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

**Step 2: Load IMDB Dataset**

In [ ]:
max_words = 20000  # vocabulary size

(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=max_words)

print("Training samples:", len(X_train))
print("Testing samples:", len(X_test))

Training samples: 25000
Testing samples: 25000


**Step 3: Pad Sequences**

In [ ]:
max_len = 300

X_train = pad_sequences(X_train, maxlen=max_len)
X_test = pad_sequences(X_test, maxlen=max_len)

**Step 4: Build LSTM Model**

In [ ]:
model = Sequential([
    Embedding(input_dim=max_words, output_dim=128, input_length=max_len),
    LSTM(128),
    Dropout(0.3),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

Model: "sequential_12"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_12 (Embedding)        │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_12 (Dropout)            │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_12 (Dense)                │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

**Step 5: Train Model**

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 217s 679ms/step - accuracy: 0.7789 - loss: 0.4579 - val_accuracy: 0.8598 - val_loss: 0.3406
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 212s 678ms/step - accuracy: 0.8667 - loss: 0.3233 - val_accuracy: 0.8502 - val_loss: 0.3663
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 213s 681ms/step - accuracy: 0.9052 - loss: 0.2430 - val_accuracy: 0.8310 - val_loss: 0.3831
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 217s 693ms/step - accuracy: 0.9352 - loss: 0.1746 - val_accuracy: 0.7742 - val_loss: 0.4836
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 258s 678ms/step - accuracy: 0.9622 - loss: 0.1112 - val_accuracy: 0.8590 - val_loss: 0.3994
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 212s 676ms/step - accuracy: 0.9801 - loss: 0.0643 - val_accuracy: 0.8590 - val_loss: 0.5072
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 263s 680ms/step - accuracy: 0.9844 - loss: 0.0511 - val_accuracy: 0.8558 - val_loss: 0.5502
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 212s 677ms/step - accuracy: 0.9884 -

**Step 6: Evaluate**

In [ ]:
loss, accuracy = model.evaluate(X_test, y_test)
print(f"Test Accuracy: {accuracy*100:.2f}%")

782/782 ━━━━━━━━━━━━━━━━━━━━ 24s 31ms/step - accuracy: 0.8645 - loss: 0.3522
Test Accuracy: 86.45%
